In [ ]:
# Import libraries
from pyspark.sql import SparkSession
from lib.helpers import get_run_date
from lib.job_control import get_max_insert_ts, insert_log
from pyspark.sql.functions import current_timestamp, lit, expr
from lib.aws_s3 import warehouse_dir

In [ ]:
# Create the SparkSession
"""
spark = (
    SparkSession.builder
    .appName("13 - Part Staging")
    .config("spark.sql.warehouse.dir", warehouse_dir)
    .enableHiveSupport()
    .getOrCreate()
)

spark
"""

In [ ]:
# Job parameters
now_ts = spark.range(1).select(current_timestamp().alias("now")).first()["now"]
run_date = get_run_date()
table = "part"
schema_name = "stg"
table_name = f"dim_{table}_{schema_name}"
table_full_name = f"{schema_name}.{table_name}"
landing_table_full_name = f"lnd.dim_{table}_lnd"

print("****************************************")
print("13 - Part Staging")
print("****************************************")
print(f"now_ts = {now_ts}")
print(f"run_date = {run_date}")
print(f"table = {table}")
print(f"schema_name = {schema_name}")
print(f"table_name = {table_name}")
print(f"table_full_name = {table_full_name}")
print(f"landing_table_full_name = {landing_table_full_name}")
print("")

In [ ]:
# Get max insert_ts for {schema_name}.{table_name} table from prd.job_control
max_insert_ts = get_max_insert_ts(spark, schema_name, table_name)
print(f"Max insert_ts in {table_full_name}: {max_insert_ts}\n")

In [ ]:
# Get latest data from landing
df_lnd = (
    spark
    .table(landing_table_full_name)
    .where(f"insert_ts > to_timestamp('{max_insert_ts}')")
)

In [ ]:
print(f"df_lnd: {df_lnd.count()} records read from {landing_table_full_name}")
df_lnd.show()
df_lnd.printSchema()

In [ ]:
# De-dupe data based on natural key
df_deduped = (
    df_lnd.withColumn("_rnk", expr(f"row_number() over (partition by part_code order by insert_ts desc)"))
    .where("_rnk = 1").drop("_rnk")
)

In [ ]:
print(f"df_deduped: {df_deduped.count()} records after de-duping\n")

In [ ]:
# Format landing data and add audit column
df_stg = (
    df_deduped
    .withColumn("insert_ts", lit(now_ts))
)

In [ ]:
print("Updated audit column")
print(f"df_stg: {df_stg.count()} records")
df_stg.show()
df_stg.printSchema()

In [ ]:
# Load data to stg schema in overwrite mode
df_stg.write.format("delta").mode("overwrite").saveAsTable(table_full_name)

In [ ]:
print(f"{df_stg.count()} records written to {table_full_name} (mode=overwrite)")

In [ ]:
print(f"Data loaded in this run:")
spark.table(f'{schema_name}.dim_{table}_{schema_name}').show()

In [ ]:
# Add entry to prd.job_control
if insert_log(spark, schema_name, table_name, now_ts, run_date):
    print(f"{schema_name}.{table_name} entry added to prd.job_control")

In [ ]:
# Review added entry
spark.sql(f"select * from prd.job_control where table_name = '{table_name}' order by insert_ts desc limit 1").show(truncate=False)

In [ ]:
# Stop spark session
#spark.stop()